## Vessel segmentation using a U-Net model
This notebook demonstrates the use of the vseg vessel segmentation network included in the toolkit. The model is trained on LSFM data.

In [ ]:
# Import the necessary libraries
import os
import shutil

import numpy as np
from liom_toolkit.segmentation.vseg.model import VsegModel
from liom_toolkit.segmentation.vseg.prediction import predict_one
from liom_toolkit.utils import extract_zarr_to_image, save_label_to_zarr
from liom_toolkit.utils.io import generate_label_color_dict_mask
from tqdm.auto import tqdm
import dask.array as da

### Prepare the zarr volume
The individual slices need to be extracted from the zarr volume and saved as individual png files. The following code demonstrates how to do this.

In [ ]:
zarr_file = ""
png_dir = ""
# channel selects the channel to extract when the volume is 4D (c, z, y, x).
extract_zarr_to_image(zarr_file, png_dir, channel=0)

### Run the model
The following code downloads the pre-trained model and runs it for the entire folder

In [ ]:
# Set the device, 'cuda' if the machine has an Nvidia GPU and CUDA installed, otherwise 'cpu'
device = "cuda"
# Model load. pretrained=True downloads weights from a wandb model artifact;
# pretrained_artifact is the wandb path (requires `wandb login` + access to the
# liom-lab project). Use pretrained=False to train from scratch instead.
model = VsegModel(pretrained=True, pretrained_artifact="liom-lab/model-registry/Vessel Segmentation:latest", device=device)

In [ ]:
dir_path = png_dir
output_path = ""
normalization = True
patching = False
stride = None
width = None

In [ ]:
# Run the model
vessel_stack = []
for images in tqdm(os.listdir(dir_path), desc="Processing images"):
    image_path = os.path.join(dir_path, images)
    prediction = predict_one(model=model, img_path=image_path, save_path=output_path,
                             norm=normalization, dev=device, patching=patching)
    prediction_dask = da.from_array(prediction, chunks=(128, 128))
    vessel_stack.append(prediction_dask)
volume = da.stack(vessel_stack, axis=0)

### Reassemble the slices to save as zarr
Next, we need to save the files back to a zarr volume. The following code demonstrates how to do this.

In [ ]:
# Make the volume binary
volume[volume > 0] = 1

# Save the volume to zarr
colour_dict = generate_label_color_dict_mask()
save_label_to_zarr(volume, zarr_file, colour_dict, "vessels", resolution_level=0)

### Cleanup
Finally, we can clean up the png files that were created.

In [ ]:
shutil.rmtree(png_dir)